In [0]:
# =============================================================================
# NOTEBOOK  : gold_dim_supplier
# PURPOSE   : silver.supplier_master → gold.dim_supplier
# LAYER     : Gold (dimension table)
# FREQUENCY : Weekly (after silver_supplier_master completes)
# WRITE     : Full overwrite — dim table, small (200 rows), always current state
# NOTE      : baseline_risk_tier is derived from master data attributes
#             actual_risk_tier (based on real PO performance) lives in
#             gold.supplier_performance_fact
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit
from utils.schema_registry import GOLD_DIM_SUPPLIER_SCHEMA
 
from pyspark.sql.functions import current_timestamp, lit, col, when
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["silver_supplier_master"]
TARGET_TABLE = cfg["tables"]["gold_dim_supplier"]
PIPELINE     = "gold_dim_supplier"

In [0]:
# ── Read + Transform + Write ─────────────────────────────────────────
run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
try:
    df = (
        spark.read.table(SOURCE_TABLE)
        .filter(col("is_current") == True)
    )
 
    rows_read = df.count()
    print(f"[READ] {rows_read} current supplier records")
 
    df = (
        df
        .withColumn("baseline_risk_tier",
            when(col("on_time_delivery_pct") >= 90, "Low")
            .when(col("on_time_delivery_pct") >= 80, "Medium")
            .otherwise("High"))
        .withColumn("performance_band",
            when(col("performance_rating") >= 4.5, "Excellent")
            .when(col("performance_rating") >= 3.5, "Good")
            .otherwise("Average"))
        .withColumn("_gold_processed_at", current_timestamp())
        .withColumn("pipeline_run_id",    lit(run_id))
        .select([f.name for f in GOLD_DIM_SUPPLIER_SCHEMA.fields])
    )
 
    rows_written = df.count()
 
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "false")
        .saveAsTable(TARGET_TABLE)
    )
 
    print(f"[DONE] {rows_written} rows written to {TARGET_TABLE}")
 
    end_audit(spark, PROJECT_ROOT, run_id, "SUCCESS",
              rows_read=rows_read, rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
    raise

In [0]:
# ── Verify ───────────────────────────────────────────────────────────
from pyspark.sql.functions import col
 
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()).limit(1) \
    .select("status", "rows_read", "rows_written").show(truncate=False)
 
print("Row count:", spark.read.table(TARGET_TABLE).count())
spark.read.table(TARGET_TABLE).groupBy("baseline_risk_tier", "performance_band").count().show()